In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

RADAR_FIRSTS_PD = pd.DataFrame({
    "name": ["Berlin", "Munich"],
    "dwd_id": ["BER", "MUC"],
    "wmo_id": [10384, 10866],
    "coordinates_wgs84_text": ["txt1", "txt2"],
    "coordinates_wgs84": ["52,50N", "48,10N"],
    "coordinates_gauss": ["gauss1", "gauss2"],
    "altitude": [45, 519],
})
RADAR_SECONDS_PD = pd.DataFrame({
    "coordinates_wgs84": ["13,40E", "11,60E"],
})
RADAR_FIRSTS_PL = pl.from_pandas(RADAR_FIRSTS_PD)
RADAR_SECONDS_PL = pl.from_pandas(RADAR_SECONDS_PD)
RADAR_RECORDS_PD = pd.DataFrame({
    "dwd_id": ["BER", "MUC"],
    "name": ["Berlin", "Munich"],
    "latitude": [52.5, 48.1],
    "longitude": [13.4, 11.6],
})
RADAR_RECORDS_PL = pl.from_pandas(RADAR_RECORDS_PD)

# --- radar_sites_cols ---
FIX_RADAR_SITES_COLS_SECONDS_PD = RADAR_SECONDS_PD
FIX_RADAR_SITES_COLS_SECONDS_PL = RADAR_SECONDS_PL

# --- radar_sites_dicts ---
df = RADAR_RECORDS_PD

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_radar_sites_cols(seconds, data=None):
    if data is None:
        data = pd.DataFrame({"name":["site1"],"state":["BY"],"coordinates_wgs84_text":["a"],"coordinates_gauss":["b"]})
    data = data.drop(labels=["coordinates_wgs84_text", "coordinates_gauss"], axis="columns")
    data = data.rename(columns={"coordinates_wgs84": "latitude"})
    data.insert(4, "longitude", seconds["coordinates_wgs84"].values)
    data = data.reset_index(drop=True)
    for column in ["latitude", "longitude"]:
        data[column] = data[column].apply(lambda x: x.strip("NE").replace(",", ".")).apply(float)
    for column in ["wmo_id", "altitude"]:
        data[column] = data[column].apply(int)
    return data

def before_radar_sites_dicts():
    result = {}
    for item in df.to_dict(orient="records"):
        key = item["dwd_id"]
        value = item
        result[key] = value
    return result



In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_radar_sites_cols(seconds, data=None):
    if data is None:
        data = pl.DataFrame({"name":["site1"],"state":["BY"],"coordinates_wgs84_text":["a"],"coordinates_gauss":["b"]})
    data = data.drop(["coordinates_wgs84_text", "coordinates_gauss"])
    data = data.rename({"coordinates_wgs84": "latitude"})
    data = data.with_columns(pl.Series("longitude", seconds["coordinates_wgs84"]))
    cols = data.columns
    cols.remove("longitude")
    cols.insert(4, "longitude")
    data = data.select(cols)
    for column in ["latitude", "longitude"]:
        data = data.with_columns(
            pl.col(column).cast(pl.Utf8).str.strip_chars("NE").str.replace(",", ".").cast(pl.Float64).alias(column)
        )
    for column in ["wmo_id", "altitude"]:
        data = data.with_columns(pl.col(column).cast(pl.Int64).alias(column))
    return data

def gen_radar_sites_dicts():
    result = {}
    for item in df.to_dicts():
        key = item["dwd_id"]
        value = item
        result[key] = value
    return result

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: radar_sites_dicts ===

def _call_radar_dicts(frame, func):
    global df
    old_df = df
    try:
        df = frame
        return func()
    finally:
        df = old_df

def _records_from_frame(frame):
    return frame.to_dict(orient="records") if isinstance(frame, pd.DataFrame) else frame.to_dicts()

# L1 smoke – generated
try:
    _old_df = df
    df = RADAR_RECORDS_PL
    _r = gen_radar_sites_dicts()
    print("✅ L1 smoke gen_radar_sites_dicts: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_radar_sites_dicts: {type(_e).__name__}: {_e}")
finally:
    df = _old_df

# L1 smoke – before
try:
    _old_df = df
    df = RADAR_RECORDS_PD
    _rb = before_radar_sites_dicts()
    print("✅ L1 smoke before_radar_sites_dicts: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_radar_sites_dicts: {type(_e).__name__}: {_e}")
finally:
    df = _old_df

# L2 behavioral equivalence: both build a {dwd_id: row_dict} mapping.
try:
    _old_df = df
    df = RADAR_RECORDS_PD
    _rb = before_radar_sites_dicts()
    df = RADAR_RECORDS_PL
    _rg = gen_radar_sites_dicts()
    assert _rb == _rg, f"{_rb!r} != {_rg!r}"
    print("✅ L2 equivalence radar_sites_dicts: MATCH")
except Exception as _e:
    print(f"❌ L2 equivalence radar_sites_dicts: setup error — {type(_e).__name__}: {_e}")
finally:
    df = _old_df

# L3 — empty records still iterate without crash and both return an empty mapping.
try:
    _old_df = df
    df = RADAR_RECORDS_PD.head(0)
    _rb = before_radar_sites_dicts()
    df = RADAR_RECORDS_PL.head(0)
    _rg = gen_radar_sites_dicts()
    assert _rb == _rg == {}, f"{_rb!r} != {_rg!r}"
    print("✅ L3 radar_sites_dicts empty records: MATCH")
except Exception as _e:
    print(f"❌ L3 radar_sites_dicts empty records: {type(_e).__name__}: {_e}")
finally:
    df = _old_df
